# Stage 6: Submission 診斷與推薦

掃描前面 stages 的 validation predictions 與 test submissions，比較 validation score、group stability 與 test distribution risk。

預設 Drive root：`/content/drive/MyDrive/VeriPromiseESG2026`。


## 0. Colab 執行環境與路徑檢查

請先執行下一個 setup cell。它會安裝套件、掛載 Google Drive、檢查 GPU、建立輸出資料夾，並驗證必要輸入檔是否存在。


In [ ]:
# Colab 啟動區塊：安裝套件、掛載 Drive、檢查 GPU、驗證必要輸入檔。
!pip -q install pandas numpy scikit-learn openpyxl

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"
STAGE_NAME = "stage6_submission_diagnostic_optional"
STAGE_DISPLAY_NAME = "Stage 6 - Submission 診斷"
OUTPUT_DIR = f"{OUTPUT_ROOT}/{STAGE_NAME}"
OUT_DIR = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

REQUIRED_RELATIVE_PATHS = [
    "data/vpesg4k_train_1000.json",
    "data/vpesg4k_valdata_1000.json",
    "data/vpesg4k_testdata_2000.json",
]

def stage_log(label, value):
    print(f"[{STAGE_NAME}] {label}: {value}")

def require_files(relative_paths):
    missing = []
    for rel in relative_paths:
        path = f"{BASE_DIR}/{rel}"
        if not os.path.exists(path):
            missing.append(path)
    if missing:
        msg = "缺少必要輸入檔。請先執行前一個 Stage，或將資料放到 Google Drive：\n" + "\n".join(missing)
        raise FileNotFoundError(msg)

require_files(REQUIRED_RELATIVE_PATHS)
stage_log("Stage", STAGE_DISPLAY_NAME)
stage_log("BASE_DIR", BASE_DIR)
stage_log("DATA_DIR", DATA_DIR)
stage_log("OUTPUT_DIR", OUTPUT_DIR)

# 完整訓練建議使用 Colab A100 或同級 GPU。
try:
    gpu_names = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_names = list(gpu_names)
    stage_log("GPU", gpu_names)
    if not any("A100" in str(name) for name in gpu_names):
        stage_log("WARNING", "建議使用 A100；非 A100 runtime 可能需要更長時間。")
except Exception as exc:
    stage_log("WARNING", f"無法取得 GPU 資訊，請確認 Colab runtime 已啟用 GPU。{exc}")


## 輸入輸出契約

Stage 顯示名稱：`Stage 6 - Submission 診斷`

Google Drive 輸出資料夾：

```text
/content/drive/MyDrive/VeriPromiseESG2026/outputs/stage6_submission_diagnostic_optional/
```

必要輸入檔：

- `data/vpesg4k_train_1000.json`
- `data/vpesg4k_valdata_1000.json`
- `data/vpesg4k_testdata_2000.json`

主要輸出檔：

- `stage6_version_summary.csv`
- `stage6_test_distribution_risk.csv`
- `stage6_recommendation.csv`
- `stage6_submission_diagnostic.xlsx`
- `stage6_report.md`

若必要輸入不存在，第一個 setup cell 會直接列出缺少的路徑並停止。


## Stage 主要流程

本節不訓練模型，主要用於彙整前面 stages 的 val/test 輸出並產生提交建議。


## 1. 路徑與基本設定

In [ ]:

import os
import json
import re
import glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage6_submission_diagnostic_optional"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

VERSION_DIRS = {
    "v34": f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend",
    "v35": f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration",
    "stage4_conservative": f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration",
    "stage4": f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration",
    "stage5_aug": f"{BASE_DIR}/outputs/stage5_model_bank_final_submission",
}

print("OUT_DIR:", OUT_DIR)
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(os.path.exists(p), p)

for name, d in VERSION_DIRS.items():
    print(name, os.path.exists(d), d)


## 2. 讀取資料

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

# IMPORTANT:
# Prediction CSVs sometimes save id as int64 while JSON may load id as object/string.
# For all merge / alignment steps, force id to string.
for df in [train_df, val_df, test_df]:
    df["id"] = df["id"].astype(str)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nVal columns:", val_df.columns.tolist())

print("\nVal label distributions:")
for t in TASKS:
    print(t, val_df[t].value_counts().to_dict())

print("\nVal group counts:")
for col in ["company", "pdf_url", "company_source"]:
    if col in val_df.columns:
        print(col, val_df[col].nunique(), val_df[col].value_counts().head().to_dict())


## 3. Official scorer

In [ ]:

def apply_hierarchy_to_pred_df(pred_df):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        pd.Series(y_true).astype(str).values,
        pd.Series(y_pred).astype(str).values,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    pred_df = apply_hierarchy_to_pred_df(pred_df)
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def add_error_cols(true_df, pred_df):
    out = true_df[["id"] + [c for c in ["data", "company", "pdf_url", "company_source", "page_number"] if c in true_df.columns]].copy()
    for t in TASKS:
        out[f"true_{t}"] = true_df[t].astype(str).values
        out[f"pred_{t}"] = pred_df[t].astype(str).values
        out[f"err_{t}"] = out[f"true_{t}"] != out[f"pred_{t}"]
    out["any_error"] = out[[f"err_{t}" for t in TASKS]].any(axis=1)
    return out


## 4. 自動尋找各版本 val predictions

In [ ]:

def normalize_version_name(path):
    name = Path(path).name
    name = name.replace("_val1000_predictions.csv", "")
    name = name.replace("_val_predictions.csv", "")
    name = name.replace("_predictions.csv", "")
    return name

def find_val_prediction_files():
    files = []
    for version, d in VERSION_DIRS.items():
        if not os.path.exists(d):
            continue
        patterns = [
            f"{d}/*val1000_predictions.csv",
            f"{d}/*val_predictions.csv",
            f"{d}/*_predictions.csv",
        ]
        for pat in patterns:
            for p in glob.glob(pat):
                if "error_analysis" in p or "confusion" in p:
                    continue
                try:
                    df0 = pd.read_csv(p, nrows=3)
                except Exception:
                    continue
                cols = set(df0.columns)
                # prediction files could be pred_* columns or direct task columns
                has_pred_cols = all(f"pred_{t}" in cols for t in TASKS)
                has_direct_cols = all(t in cols for t in TASKS)
                if has_pred_cols or has_direct_cols:
                    files.append((version, normalize_version_name(p), p))
    # de-duplicate
    seen = set()
    out = []
    for item in files:
        if item[2] not in seen:
            out.append(item)
            seen.add(item[2])
    return out

found_files = find_val_prediction_files()
print("Found val prediction files:", len(found_files))
for version, name, p in found_files:
    print(version, "=>", name, ":", p)


In [ ]:

# 可以手動指定你最關心的版本名稱。
# 如果某個檔案不存在，會自動跳過。

MANUAL_VERSION_FILES = {
    # v34 / base
    "v34_base_from_v35": f"{VERSION_DIRS['v35']}/stage4_t2_base_stage3_val1000_predictions.csv",
    "v34_base_from_stage4_conservative": f"{VERSION_DIRS['stage4_conservative']}/stage4_conservative_base_stage3_val1000_predictions.csv",
    "v34_base_from_v36": f"{VERSION_DIRS['stage4']}/stage4_base_v34_val1000_predictions.csv",
    "v37_base_v34": f"{VERSION_DIRS['stage5_aug']}/stage5_aug_base_v34_val1000_predictions.csv",

    # v35
    "stage4_t2_best_val": f"{VERSION_DIRS['v35']}/stage4_t2_best_val_val1000_predictions.csv",
    "stage4_t2_replace": f"{VERSION_DIRS['v35']}/stage4_t2_replace_val1000_predictions.csv",

    # stage4_conservative
    "stage4_conservative_best_val": f"{VERSION_DIRS['stage4_conservative']}/stage4_conservative_best_val_val1000_predictions.csv",
    "stage4_distribution_matched": f"{VERSION_DIRS['stage4_conservative']}/stage4_distribution_matched_val1000_predictions.csv",

    # v36
    "v36_best_val": f"{VERSION_DIRS['stage4']}/stage4_best_val_val1000_predictions.csv",
    "v36_safe": f"{VERSION_DIRS['stage4']}/stage4_safe_val1000_predictions.csv",
    "v36_distribution_matched": f"{VERSION_DIRS['stage4']}/stage4_distribution_matched_val1000_predictions.csv",

    # v37
    "stage5_aug_best_val": f"{VERSION_DIRS['stage5_aug']}/stage5_aug_best_val_val1000_predictions.csv",
    "stage5_aug_safe": f"{VERSION_DIRS['stage5_aug']}/stage5_aug_safe_val1000_predictions.csv",
    "stage5_aug_distribution_matched": f"{VERSION_DIRS['stage5_aug']}/stage5_aug_distribution_matched_val1000_predictions.csv",
    "stage5_aug_best_t2_only": f"{VERSION_DIRS['stage5_aug']}/stage5_aug_best_t2_only_val1000_predictions.csv",
    "stage5_aug_best_t4_only": f"{VERSION_DIRS['stage5_aug']}/stage5_aug_best_t4_only_val1000_predictions.csv",
}

# 合併 automatic + manual
version_files = {}
for version, name, p in found_files:
    key = name
    version_files[key] = p

for key, p in MANUAL_VERSION_FILES.items():
    if os.path.exists(p):
        version_files[key] = p

print("Selected version files:", len(version_files))
for k, p in sorted(version_files.items()):
    print(k, ":", p)


## 5. 讀取 predictions 並重算 fixed val 分數

In [ ]:

def load_pred_file(path):
    df = pd.read_csv(path)

    if "id" not in df.columns:
        raise ValueError(f"no id column: {path}")

    # IMPORTANT:
    # unify id type before merge, otherwise pandas may raise:
    # ValueError: You are trying to merge on object and int64 columns for key 'id'
    df["id"] = df["id"].astype(str)

    pred = pd.DataFrame()
    pred["id"] = df["id"].astype(str).values

    if all(f"pred_{t}" in df.columns for t in TASKS):
        for t in TASKS:
            pred[t] = df[f"pred_{t}"].astype(str).replace({"nan": "N/A"}).values
    elif all(t in df.columns for t in TASKS):
        for t in TASKS:
            pred[t] = df[t].astype(str).replace({"nan": "N/A"}).values
    else:
        raise ValueError(f"missing task prediction columns: {path}")

    # Align to val_df order.
    # val_df['id'] was already forced to string in the loading cell.
    key_df = val_df[["id"]].copy()
    key_df["id"] = key_df["id"].astype(str)

    pred = key_df.merge(pred, on="id", how="left")

    if pred[TASKS].isna().any().any():
        missing = pred[pred[TASKS].isna().any(axis=1)]["id"].tolist()[:10]
        raise ValueError(f"missing prediction rows for ids: {missing}")

    pred = apply_hierarchy_to_pred_df(pred)
    return pred

preds = {}
summary_rows = []

for name, path in sorted(version_files.items()):
    try:
        pred = load_pred_file(path)
        metrics = calc_metrics(val_df, pred)
        preds[name] = pred
        row = {"version": name, "path": path, **metrics}
        for t in TASKS:
            row[f"pred_dist_{t}"] = json.dumps(pred[t].value_counts().to_dict(), ensure_ascii=False)
        summary_rows.append(row)
    except Exception as e:
        print("[SKIP]", name, path, "=>", repr(e))

if len(summary_rows) == 0:
    raise RuntimeError(
        "No prediction files were loaded. Check id alignment and prediction CSV paths. "
        "Most commonly this is caused by id dtype mismatch; this fixed notebook casts all ids to string."
    )

summary_df = pd.DataFrame(summary_rows).sort_values("weighted_score", ascending=False)
summary_df.to_csv(f"{OUT_DIR}/stage6_version_summary.csv", index=False, encoding="utf-8-sig")

display(summary_df)
print("loaded versions:", list(preds.keys()))


## 6. 選 baseline 與候選版本

In [ ]:

# baseline 選擇：
# 優先 v34_base_from_v35 / v34_base_from_v36 / v37_base_v34 / score最接近 v34 的版本
preferred_baselines = [
    "v34_base_from_v35",
    "v34_base_from_v36",
    "v37_base_v34",
    "v34_base_from_stage4_conservative",
]

baseline_version = None
for b in preferred_baselines:
    if b in preds:
        baseline_version = b
        break

if baseline_version is None:
    # fallback: score最低的主要 base-like version
    baseline_version = summary_df.sort_values("weighted_score").iloc[-1]["version"]

print("baseline_version:", baseline_version)
baseline_pred = preds[baseline_version]
baseline_metrics = calc_metrics(val_df, baseline_pred)
print_metrics(baseline_metrics, f"baseline: {baseline_version}")

# 預設關注版本：若存在就納入，否則用所有版本
focus_versions = [
    baseline_version,
    "stage4_t2_best_val",
    "stage4_distribution_matched",
    "stage4_conservative_best_val",
    "v36_best_val",
    "v36_safe",
    "stage5_aug_best_val",
    "stage5_aug_safe",
    "stage5_aug_distribution_matched",
    "stage5_aug_best_t2_only",
    "stage5_aug_best_t4_only",
]
focus_versions = [v for v in focus_versions if v in preds]

if len(focus_versions) < 3:
    focus_versions = summary_df["version"].tolist()

print("focus_versions:", focus_versions)


## 7. Group fold diagnostic：company / pdf_url / source / page bucket

In [ ]:

def build_eval_df(pred, version):
    eval_df = add_error_cols(val_df, pred)
    eval_df["version"] = version
    return eval_df

def safe_group_col(df, col):
    if col in df.columns:
        return df[col].fillna("MISSING").astype(str).values
    return None

def page_bucket_series(df):
    if "page_number" not in df.columns:
        return pd.Series(["MISSING"] * len(df))
    p = pd.to_numeric(df["page_number"], errors="coerce").fillna(-1).astype(int)
    # 每 25 頁一桶
    bucket = (p // 25) * 25
    return bucket.astype(str) + "-" + (bucket + 24).astype(str)

def calc_subset_metric(version, indices):
    pred_sub = preds[version].iloc[indices].reset_index(drop=True)
    true_sub = val_df.iloc[indices].reset_index(drop=True)
    return calc_metrics(true_sub, pred_sub)

def groupfold_diagnostic(group_name, groups, versions, n_splits=5):
    groups = pd.Series(groups).fillna("MISSING").astype(str).values
    unique_n = len(pd.unique(groups))
    if unique_n < 2:
        print(f"[SKIP] {group_name}: only {unique_n} unique groups")
        return pd.DataFrame()

    n_splits = min(n_splits, unique_n)

    gkf = GroupKFold(n_splits=n_splits)
    rows = []

    dummy_X = np.zeros(len(val_df))
    dummy_y = np.zeros(len(val_df))

    for fold, (_, test_idx) in enumerate(gkf.split(dummy_X, dummy_y, groups=groups), start=1):
        fold_groups = sorted(pd.unique(groups[test_idx]).tolist())
        for version in versions:
            m = calc_subset_metric(version, test_idx)
            row = {
                "group_type": group_name,
                "fold": fold,
                "version": version,
                "n": len(test_idx),
                "n_groups": len(fold_groups),
                "groups": "|".join(map(str, fold_groups[:20])),
                **m,
            }
            rows.append(row)

    return pd.DataFrame(rows)

group_sources = {}

if "company" in val_df.columns:
    group_sources["company"] = val_df["company"].fillna("MISSING").astype(str)

if "pdf_url" in val_df.columns:
    group_sources["pdf_url"] = val_df["pdf_url"].fillna("MISSING").astype(str)

if "company_source" in val_df.columns:
    group_sources["company_source"] = val_df["company_source"].fillna("MISSING").astype(str)

group_sources["page_bucket_25"] = page_bucket_series(val_df)

groupfold_dfs = []
for group_name, groups in group_sources.items():
    df_g = groupfold_diagnostic(group_name, groups, focus_versions, n_splits=5)
    if len(df_g):
        groupfold_dfs.append(df_g)

groupfold_df = pd.concat(groupfold_dfs, ignore_index=True) if groupfold_dfs else pd.DataFrame()
groupfold_df.to_csv(f"{OUT_DIR}/stage6_groupfold_scores.csv", index=False, encoding="utf-8-sig")

display(groupfold_df.head(30))


## 8. Group fold summary：看每個版本跨 group fold 是否穩

In [ ]:

def summarize_groupfold(groupfold_df, baseline_version):
    rows = []
    if groupfold_df.empty:
        return pd.DataFrame()

    for group_type in groupfold_df["group_type"].unique():
        sub = groupfold_df[groupfold_df["group_type"] == group_type].copy()
        baseline_sub = sub[sub["version"] == baseline_version][["fold", "weighted_score"]].rename(columns={"weighted_score": "baseline_weighted_score"})

        for version in sub["version"].unique():
            vdf = sub[sub["version"] == version].merge(baseline_sub, on="fold", how="left")
            diffs = vdf["weighted_score"] - vdf["baseline_weighted_score"]

            row = {
                "group_type": group_type,
                "version": version,
                "fold_mean": vdf["weighted_score"].mean(),
                "fold_std": vdf["weighted_score"].std(ddof=0),
                "fold_min": vdf["weighted_score"].min(),
                "fold_max": vdf["weighted_score"].max(),
                "diff_vs_baseline_mean": diffs.mean(),
                "diff_vs_baseline_std": diffs.std(ddof=0),
                "win_folds_vs_baseline": int((diffs > 0).sum()),
                "tie_folds_vs_baseline": int((diffs == 0).sum()),
                "loss_folds_vs_baseline": int((diffs < 0).sum()),
                "n_folds": len(vdf),
            }

            # task means
            for k in [
                "promise_status_f1_yes",
                "verification_timeline_macro_f1",
                "evidence_status_f1_yes",
                "evidence_quality_macro_f1",
            ]:
                row[f"{k}_fold_mean"] = vdf[k].mean()
                row[f"{k}_fold_std"] = vdf[k].std(ddof=0)

            rows.append(row)

    return pd.DataFrame(rows).sort_values(["group_type", "fold_mean"], ascending=[True, False])

groupfold_summary_df = summarize_groupfold(groupfold_df, baseline_version)
groupfold_summary_df.to_csv(f"{OUT_DIR}/stage6_groupfold_summary.csv", index=False, encoding="utf-8-sig")

display(groupfold_summary_df)


## 9. Per-group metrics：找出提升是否集中在少數公司

In [ ]:

def per_group_metrics(group_type, group_values, versions, min_n=8):
    groups = pd.Series(group_values).fillna("MISSING").astype(str).values
    rows = []

    for g in sorted(pd.unique(groups)):
        idx = np.where(groups == g)[0]
        if len(idx) < min_n:
            continue

        for version in versions:
            m = calc_subset_metric(version, idx)
            rows.append({
                "group_type": group_type,
                "group": g,
                "version": version,
                "n": len(idx),
                **m,
            })

    return pd.DataFrame(rows)

per_group_dfs = []
for group_name, groups in group_sources.items():
    min_n = 8 if group_name != "pdf_url" else 5
    df_pg = per_group_metrics(group_name, groups, focus_versions, min_n=min_n)
    if len(df_pg):
        per_group_dfs.append(df_pg)

per_group_df = pd.concat(per_group_dfs, ignore_index=True) if per_group_dfs else pd.DataFrame()
per_group_df.to_csv(f"{OUT_DIR}/stage6_per_group_metrics.csv", index=False, encoding="utf-8-sig")

display(per_group_df.head(30))

# 對 baseline 做每個 group 的差分
if not per_group_df.empty:
    base_pg = per_group_df[per_group_df["version"] == baseline_version][["group_type", "group", "weighted_score"]].rename(columns={"weighted_score": "baseline_weighted_score"})
    per_group_diff_df = per_group_df.merge(base_pg, on=["group_type", "group"], how="left")
    per_group_diff_df["diff_vs_baseline"] = per_group_diff_df["weighted_score"] - per_group_diff_df["baseline_weighted_score"]
    per_group_diff_df.to_csv(f"{OUT_DIR}/stage6_per_group_diff.csv", index=False, encoding="utf-8-sig")
    display(per_group_diff_df.sort_values("diff_vs_baseline", ascending=False).head(30))
else:
    per_group_diff_df = pd.DataFrame()


## 10. Bootstrap by groups：版本提升是否穩定？

In [ ]:

def bootstrap_group_scores(group_values, versions, baseline_version, n_boot=500, group_frac=0.65, random_state=42):
    rng = np.random.default_rng(random_state)
    groups = pd.Series(group_values).fillna("MISSING").astype(str).values
    unique_groups = np.array(sorted(pd.unique(groups)))
    n_sample = max(1, int(len(unique_groups) * group_frac))

    rows = []
    for b in range(n_boot):
        sampled_groups = rng.choice(unique_groups, size=n_sample, replace=True)
        mask = np.isin(groups, sampled_groups)
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue

        base_score = calc_subset_metric(baseline_version, idx)["weighted_score"]

        for version in versions:
            m = calc_subset_metric(version, idx)
            rows.append({
                "bootstrap_id": b,
                "version": version,
                "n": len(idx),
                "score": m["weighted_score"],
                "baseline_score": base_score,
                "diff_vs_baseline": m["weighted_score"] - base_score,
            })

    return pd.DataFrame(rows)

bootstrap_dfs = []

for group_name in ["company", "pdf_url", "company_source"]:
    if group_name in group_sources:
        print("bootstrap:", group_name)
        df_bs = bootstrap_group_scores(
            group_sources[group_name],
            focus_versions,
            baseline_version,
            n_boot=500,
            group_frac=0.65,
            random_state=123,
        )
        df_bs["group_type"] = group_name
        bootstrap_dfs.append(df_bs)

bootstrap_df = pd.concat(bootstrap_dfs, ignore_index=True) if bootstrap_dfs else pd.DataFrame()
bootstrap_df.to_csv(f"{OUT_DIR}/stage6_bootstrap_group_scores.csv", index=False, encoding="utf-8-sig")

if not bootstrap_df.empty:
    bs_summary = bootstrap_df.groupby(["group_type", "version"]).agg(
        score_mean=("score", "mean"),
        score_std=("score", "std"),
        diff_mean=("diff_vs_baseline", "mean"),
        diff_std=("diff_vs_baseline", "std"),
        win_rate=("diff_vs_baseline", lambda x: float((x > 0).mean())),
        p05_diff=("diff_vs_baseline", lambda x: float(np.quantile(x, 0.05))),
        p50_diff=("diff_vs_baseline", lambda x: float(np.quantile(x, 0.50))),
        p95_diff=("diff_vs_baseline", lambda x: float(np.quantile(x, 0.95))),
        n_boot=("diff_vs_baseline", "count"),
    ).reset_index()
    bs_summary.to_csv(f"{OUT_DIR}/stage6_bootstrap_summary.csv", index=False, encoding="utf-8-sig")
    display(bs_summary.sort_values(["group_type", "diff_mean"], ascending=[True, False]))
else:
    bs_summary = pd.DataFrame()


## 11. Test distribution risk：submission 分布是否過度偏移

In [ ]:

def find_submission_files():
    files = {}
    for version, d in VERSION_DIRS.items():
        if not os.path.exists(d):
            continue
        for p in glob.glob(f"{d}/*test2000_submission.csv"):
            key = Path(p).name.replace("_test2000_submission.csv", "")
            files[key] = p
    return files

submission_files = find_submission_files()
print("Found submissions:", len(submission_files))
for k, p in sorted(submission_files.items()):
    print(k, ":", p)

def load_submission(path):
    df = pd.read_csv(path)
    if not all(c in df.columns for c in ["id"] + TASKS):
        raise ValueError(f"bad submission: {path}")

    # IMPORTANT: unify id type before merge.
    df["id"] = df["id"].astype(str)
    key_df = test_df[["id"]].copy()
    key_df["id"] = key_df["id"].astype(str)

    pred = key_df.merge(df[["id"] + TASKS], on="id", how="left")
    for t in TASKS:
        pred[t] = pred[t].fillna("N/A").astype(str).replace({"nan": "N/A"})
    return apply_hierarchy_to_pred_df(pred)

def dist_dict(pred, task):
    return pred[task].value_counts().to_dict()

# Reference: v34 if available
ref_submission_key = None
for k in ["stage5_aug_base_v34", "stage4_base_v34", "stage4_conservative_base_stage3", "stage4_t2_base_stage3", "stage3_best_val"]:
    if k in submission_files:
        ref_submission_key = k
        break

if ref_submission_key is None and len(submission_files):
    ref_submission_key = sorted(submission_files.keys())[0]

print("ref_submission_key:", ref_submission_key)

if ref_submission_key:
    ref_sub = load_submission(submission_files[ref_submission_key])
    ref_timeline = dist_dict(ref_sub, "verification_timeline")
    ref_quality = dist_dict(ref_sub, "evidence_quality")
else:
    ref_timeline = {}
    ref_quality = {}

def calc_test_risk(pred, ref_timeline, ref_quality):
    tl = dist_dict(pred, "verification_timeline")
    q = dist_dict(pred, "evidence_quality")

    already_drop = max(0, ref_timeline.get("already", 0) - tl.get("already", 0))
    more5_inc = max(0, tl.get("more_than_5_years", 0) - ref_timeline.get("more_than_5_years", 0))
    within2_inc = max(0, tl.get("within_2_years", 0) - ref_timeline.get("within_2_years", 0))
    timeline_risk = already_drop / 120.0 + more5_inc / 120.0 + within2_inc / 25.0

    clear_inc = max(0, q.get("Clear", 0) - ref_quality.get("Clear", 0))
    not_clear_inc = max(0, q.get("Not Clear", 0) - ref_quality.get("Not Clear", 0))
    misleading_inc = max(0, q.get("Misleading", 0) - ref_quality.get("Misleading", 0))
    quality_risk = clear_inc / 200.0 + not_clear_inc / 120.0 + misleading_inc / 20.0

    return {
        "timeline_risk": timeline_risk,
        "quality_risk": quality_risk,
        "total_risk": timeline_risk + quality_risk,
        "already_drop": already_drop,
        "more5_inc": more5_inc,
        "within2_inc": within2_inc,
        "clear_inc": clear_inc,
        "not_clear_inc": not_clear_inc,
        "misleading_inc": misleading_inc,
    }

risk_rows = []
for key, path in sorted(submission_files.items()):
    try:
        sub = load_submission(path)
        risk = calc_test_risk(sub, ref_timeline, ref_quality)
        row = {
            "submission_key": key,
            "path": path,
            **risk,
        }
        for t in TASKS:
            row[f"dist_{t}"] = json.dumps(dist_dict(sub, t), ensure_ascii=False)
        risk_rows.append(row)
    except Exception as e:
        print("[SKIP]", key, repr(e))

test_risk_df = pd.DataFrame(risk_rows).sort_values("total_risk") if risk_rows else pd.DataFrame()
test_risk_df.to_csv(f"{OUT_DIR}/stage6_test_distribution_risk.csv", index=False, encoding="utf-8-sig")
display(test_risk_df.head(50))


## 12. 綜合推薦分數

In [ ]:

def pick_group_summary_metric(version, group_type="company"):
    if groupfold_summary_df.empty:
        return {}
    row = groupfold_summary_df[
        (groupfold_summary_df["version"] == version) &
        (groupfold_summary_df["group_type"] == group_type)
    ]
    if len(row) == 0:
        return {}
    return row.iloc[0].to_dict()

def pick_bootstrap_metric(version, group_type="company"):
    if bs_summary.empty:
        return {}
    row = bs_summary[
        (bs_summary["version"] == version) &
        (bs_summary["group_type"] == group_type)
    ]
    if len(row) == 0:
        return {}
    return row.iloc[0].to_dict()

def find_matching_submission_risk(version):
    # heuristic matching by substring
    if test_risk_df.empty:
        return {}
    candidates = []
    for _, row in test_risk_df.iterrows():
        key = row["submission_key"]
        if version in key or key in version:
            candidates.append(row)
    if not candidates:
        # common mapping
        mapping = {
            "stage5_aug_best_val": "stage5_aug_best_val",
            "stage5_aug_safe": "stage5_aug_safe",
            "stage5_aug_distribution_matched": "stage5_aug_distribution_matched",
            "v36_best_val": "stage4_best_val",
            "v36_safe": "stage4_safe",
            "stage4_distribution_matched": "stage4_distribution_matched",
            "stage4_t2_best_val": "stage4_t2_best_blend",
        }
        target = mapping.get(version)
        if target:
            for _, row in test_risk_df.iterrows():
                if target in row["submission_key"]:
                    candidates.append(row)
    if candidates:
        return pd.DataFrame(candidates).sort_values("total_risk").iloc[0].to_dict()
    return {}

rec_rows = []

for _, srow in summary_df.iterrows():
    version = srow["version"]
    if version not in focus_versions:
        continue

    company_g = pick_group_summary_metric(version, "company")
    company_bs = pick_bootstrap_metric(version, "company")
    sub_risk = find_matching_submission_risk(version)

    fixed_score = srow["weighted_score"]
    group_mean = company_g.get("fold_mean", np.nan)
    group_std = company_g.get("fold_std", np.nan)
    diff_mean = company_g.get("diff_vs_baseline_mean", np.nan)
    win_folds = company_g.get("win_folds_vs_baseline", np.nan)
    win_rate = company_bs.get("win_rate", np.nan)
    p05_diff = company_bs.get("p05_diff", np.nan)
    risk = sub_risk.get("total_risk", np.nan)

    # heuristic trust score:
    # - fixed score 高加分
    # - group mean 高加分
    # - std/risk 扣分
    # - bootstrap win_rate 加分
    trust_score = 0.0
    if not np.isnan(fixed_score):
        trust_score += fixed_score * 1.0
    if not np.isnan(group_mean):
        trust_score += group_mean * 0.8
    if not np.isnan(diff_mean):
        trust_score += diff_mean * 2.0
    if not np.isnan(win_rate):
        trust_score += (win_rate - 0.5) * 0.05
    if not np.isnan(group_std):
        trust_score -= group_std * 0.4
    if not np.isnan(risk):
        trust_score -= risk * 0.01

    rec_rows.append({
        "version": version,
        "fixed_weighted_score": fixed_score,
        "company_fold_mean": group_mean,
        "company_fold_std": group_std,
        "company_diff_vs_baseline_mean": diff_mean,
        "company_win_folds_vs_baseline": win_folds,
        "company_bootstrap_win_rate": win_rate,
        "company_bootstrap_p05_diff": p05_diff,
        "matched_test_risk": risk,
        "trust_score_heuristic": trust_score,
        "path": srow.get("path", ""),
    })

recommendation_df = pd.DataFrame(rec_rows).sort_values("trust_score_heuristic", ascending=False)
recommendation_df.to_csv(f"{OUT_DIR}/stage6_recommendation.csv", index=False, encoding="utf-8-sig")
display(recommendation_df)


## 13. 輸出 Excel 報告

In [ ]:

xlsx_path = f"{OUT_DIR}/stage6_submission_diagnostic.xlsx"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="version_summary", index=False)
    recommendation_df.to_excel(writer, sheet_name="recommendation", index=False)

    if not groupfold_df.empty:
        groupfold_df.to_excel(writer, sheet_name="groupfold_scores", index=False)
    if not groupfold_summary_df.empty:
        groupfold_summary_df.to_excel(writer, sheet_name="groupfold_summary", index=False)
    if not per_group_df.empty:
        per_group_df.to_excel(writer, sheet_name="per_group_metrics", index=False)
    if not per_group_diff_df.empty:
        per_group_diff_df.to_excel(writer, sheet_name="per_group_diff", index=False)
    if not bootstrap_df.empty:
        bootstrap_df.head(5000).to_excel(writer, sheet_name="bootstrap_scores_head", index=False)
    if not bs_summary.empty:
        bs_summary.to_excel(writer, sheet_name="bootstrap_summary", index=False)
    if not test_risk_df.empty:
        test_risk_df.to_excel(writer, sheet_name="test_risk", index=False)

print("saved:", xlsx_path)


## 14. Markdown 報告摘要

In [ ]:

def fmt_float(x, n=6):
    try:
        if pd.isna(x):
            return "NA"
        return f"{float(x):.{n}f}"
    except Exception:
        return str(x)

report = []
report.append("# Stage 6 Submission 診斷報告")
report.append("")
report.append("## Loaded versions")
report.append(f"- baseline_version: `{baseline_version}`")
report.append(f"- focus_versions: `{', '.join(focus_versions)}`")
report.append("")

report.append("## Fixed val1000 summary")
for _, row in summary_df.head(15).iterrows():
    report.append(f"- `{row['version']}`: weighted={fmt_float(row['weighted_score'])}, "
                  f"T2={fmt_float(row['verification_timeline_macro_f1'])}, "
                  f"T4={fmt_float(row['evidence_quality_macro_f1'])}")

report.append("")
report.append("## Recommendation heuristic")
for _, row in recommendation_df.head(15).iterrows():
    report.append(
        f"- `{row['version']}`: fixed={fmt_float(row['fixed_weighted_score'])}, "
        f"company_fold_mean={fmt_float(row['company_fold_mean'])}, "
        f"company_diff={fmt_float(row['company_diff_vs_baseline_mean'])}, "
        f"win_rate={fmt_float(row['company_bootstrap_win_rate'], 3)}, "
        f"test_risk={fmt_float(row['matched_test_risk'], 3)}, "
        f"trust={fmt_float(row['trust_score_heuristic'])}"
    )

if not groupfold_summary_df.empty:
    report.append("")
    report.append("## Company GroupFold summary")
    sub = groupfold_summary_df[groupfold_summary_df["group_type"] == "company"].sort_values("fold_mean", ascending=False)
    for _, row in sub.head(15).iterrows():
        report.append(
            f"- `{row['version']}`: fold_mean={fmt_float(row['fold_mean'])}, "
            f"fold_std={fmt_float(row['fold_std'])}, "
            f"diff_vs_base={fmt_float(row['diff_vs_baseline_mean'])}, "
            f"wins={int(row['win_folds_vs_baseline'])}/{int(row['n_folds'])}"
        )

if not bs_summary.empty:
    report.append("")
    report.append("## Company bootstrap summary")
    sub = bs_summary[bs_summary["group_type"] == "company"].sort_values("diff_mean", ascending=False)
    for _, row in sub.head(15).iterrows():
        report.append(
            f"- `{row['version']}`: diff_mean={fmt_float(row['diff_mean'])}, "
            f"win_rate={fmt_float(row['win_rate'], 3)}, "
            f"p05={fmt_float(row['p05_diff'])}, "
            f"p95={fmt_float(row['p95_diff'])}"
        )

if not test_risk_df.empty:
    report.append("")
    report.append("## Lowest test distribution risk")
    for _, row in test_risk_df.head(15).iterrows():
        report.append(
            f"- `{row['submission_key']}`: total_risk={fmt_float(row['total_risk'], 3)}, "
            f"timeline={fmt_float(row['timeline_risk'], 3)}, "
            f"quality={fmt_float(row['quality_risk'], 3)}"
        )

report.append("")
report.append("## Files")
report.append(f"- Excel: `{xlsx_path}`")
report.append(f"- Summary CSV: `{OUT_DIR}/stage6_version_summary.csv`")
report.append(f"- Recommendation CSV: `{OUT_DIR}/stage6_recommendation.csv`")

report_md = "\n".join(report)

md_path = f"{OUT_DIR}/stage6_report.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:10000])
print("saved:", md_path)
